In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import pandas as pd
import scanpy as sc
import warnings
from jax import random
import scipy.sparse as sp
warnings.filterwarnings('ignore')

import enclus

# ******** preprocess ********

sc_data = "dataset5_seq_915.h5ad"
sp_data = 'dataset5_spatial_915.h5ad'

document = 'dataset5'
rand = 0
n_splits = 5
outdir = 'Result/' + document + '/'

adata_spatial = sc.read_h5ad('datasets/sp/' + sp_data)
adata_seq = sc.read_h5ad('datasets/sc/' + sc_data)
if isinstance(adata_seq.X, sp.csr_matrix):
    adata_seq.X = adata_seq.X.toarray()

print(adata_spatial.X.max(),adata_seq.X.max())


adata_seq2 = adata_seq.copy()
sc.pp.log1p(adata_seq2)
data_seq_array = adata_seq2.X

adata_spatial2 = adata_spatial.copy()
sc.pp.log1p(adata_spatial2)
data_spatial_array = adata_spatial2.X

print(adata_spatial2.X.max(),adata_seq2.X.max())

sp_genes = np.array(adata_spatial.var_names)


def ENCLUS_impute(adata_seq2, adata_spatial2,
                n_splits=5, training_steps=6000, batch_size=1024, 
                verbose=16, init_lr=0.00001, decay_steps=5000):

    raw_shared_gene = np.array(adata_spatial2.var_names)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=rand)
    kf.get_n_splits(raw_shared_gene)

    all_pred_res = np.zeros_like(adata_spatial2.X)
    fold = 1
    for train_ind, test_ind in kf.split(raw_shared_gene):
        print(f"\n===== Fold {fold} =====")
        print(f"Number of train genes: {len(train_ind)}, Number of test genes: {len(test_ind)}")
        train_genes = raw_shared_gene[train_ind]

        sp_data_partial = adata_spatial2[:, train_genes].copy()
        enclus_model = enclus.ENCLUS(
            spatial_data=sp_data_partial,
            sc_data=adata_seq2,
            num_layers=3,
            num_neurons=1024,
            latent_dim=256,
            k_nearest=8,
            num_cov_genes=64,
            cov_genes=[],
            num_HVG=adata_seq2.shape[1],
            spatial_dist="pois",
            sc_dist="nb",
            spatial_coeff=1.0, 
            sc_coeff=1.0,    
            kl_coeff=0.1,
            n_clusters=4,
            tau=0.1,
            gamma=0.1,
            adaptive_weights=True,
            early_stopping=True,
            patience=30,
            distance_metric='euclidean',
            num_heads=8,
            head_dim=64,
        )
        enclus_model.train(
            training_steps=training_steps,
            batch_size=batch_size,
            verbose=verbose,
            init_lr=init_lr,
            decay_steps=decay_steps
        )
        enclus_model.impute_genes()
        imputed = enclus_model.spatial_data.obsm['imputation'].iloc[:, test_ind].values
        all_pred_res[:, test_ind] = imputed

        fold += 1

    return all_pred_res


40.0 49868.0
3.713572 10.817155


In [16]:
enclus_result = ENCLUS_impute(adata_seq2, adata_spatial2)
enclus_result_pd = pd.DataFrame(enclus_result, columns=sp_genes)


===== Fold 1 =====
Number of train genes: 94, Number of test genes: 24
sc_data shape and st_data shape: (15928, 118) (1277, 94)
Initializing CVAE
Finished Initializing ENCLUS
Initializing cluster centers...


 | spatial_w: 8.04 sc_w: 1.31 cov_w: 9.90 kl_w: 0.69 cluster_w: 1.59:  12%|█▏        | 743/6000 [20:27<2:24:42,  1.65s/it]


Early stopping triggered


Finished imputing missing gene for spatial data! See 'imputation' in obsm of ENCLUS.spatial_data

===== Fold 2 =====
Number of train genes: 94, Number of test genes: 24
sc_data shape and st_data shape: (15928, 118) (1277, 94)
Initializing CVAE
Finished Initializing ENCLUS
Initializing cluster centers...


 | spatial_w: 7.50 sc_w: 1.33 cov_w: 9.90 kl_w: 0.68 cluster_w: 1.63:  12%|█▏        | 743/6000 [20:41<2:26:22,  1.67s/it]

...


Early stopping triggered


Finished imputing missing gene for spatial data! See 'imputation' in obsm of ENCLUS.spatial_data

===== Fold 5 =====
Number of train genes: 95, Number of test genes: 23
sc_data shape and st_data shape: (15928, 118) (1277, 95)
Initializing CVAE
Finished Initializing ENCLUS
Initializing cluster centers...


 | spatial_w: 7.63 sc_w: 1.34 cov_w: 9.91 kl_w: 0.69 cluster_w: 1.56:  12%|█▏        | 743/6000 [20:49<2:27:22,  1.68s/it]


Early stopping triggered


Finished imputing missing gene for spatial data! See 'imputation' in obsm of ENCLUS.spatial_data


In [17]:
enclus_result_pd

,PDGFRA,DCN,FTH1,STMN1,CALM2,SPARCL1,RIMS2,SLC1A2,NELL2,CHN1,...,COL24A1,KCNT2,RXFP1,FSTL5,EGFEM1P,PCDH9,KIT,SULF1,PAX6,COL22A1
0,0.066723,0.014950,0.266491,0.027249,0.071998,0.071317,0.553861,1.317616,0.007475,0.110174,...,0.069859,0.412651,0.596217,0.165425,1.042091,0.813053,0.475792,0.389146,0.132845,0.188000
1,0.099692,0.014548,0.366559,0.022915,0.073686,0.041857,0.454382,0.612364,0.008269,0.117607,...,0.087268,0.612339,0.423632,0.123152,0.930137,0.652337,0.921249,0.316837,0.127327,0.340332
2,0.096197,0.019951,0.345409,0.016499,0.077611,0.042832,1.056700,0.639737,0.010913,0.054924,...,0.095835,1.008559,0.335982,0.108210,2.420272,0.766668,0.309221,0.637288,0.088640,0.273439
3,0.067467,0.015509,0.054579,0.014905,0.109922,0.030266,0.658558,0.348388,0.011498,0.130144,...,0.073974,0.959702,0.382645,0.109972,2.047306,1.054223,0.226891,0.181664,0.123684,0.212151
4,0.096598,0.013283,0.180406,0.016818,0.119507,0.043190,0.359945,0.586693,0.008151,0.097798,...,0.079273,0.507258,1.745379,0.075381,1.063287,0.730048,0.358768,0.390675,0.160629,0.395766
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1272,0.048433,0.009355,0.048408,0.025688,0.055364,0.041033,0.780512,0.466044,0.007097,0.148848,...,0.101064,0.642635,0.574649,0.083964,0.983657,0.923526,0.276371,0.157847,0.116233,0.195497
1273,0.158495,0.013617,0.186620,0.011626,0.288062,0.063236,0.566471,0.191666,0.007883,0.038865,...,0.135729,0.869793,1.267619,0.096024,0.834102,0.773669,0.153721,0.097369,0.135914,0.408430
1274,0.052819,0.016008,0.335306,0.030588,0.038041,0.038299,0.310833,1.705820,0.007616,0.170684,...,0.110006,0.459342,0.618231,0.185525,1.200191,0.778949,0.944967,0.736011,0.104897,0.271578
1275,0.021846,0.012606,0.594946,0.078281,0.025220,0.060536,0.262501,1.416189,0.009967,0.119726,...,0.024175,0.319921,1.080445,0.208667,0.810965,0.404756,0.578557,0.707044,0.156504,0.261966


In [21]:
enclus_result_pd.to_csv(outdir +  '/SpateCV_impute.csv',header = 1, index = 1)